In [ ]:
import polars as pl
import seaborn as sns

In [ ]:
df = pl.read_excel("KE_QoS_Report_20260211.xlsx" )
df.head()

In [ ]:
kenya = df[0:57].with_row_index("idx", offset=1)
nairobi = df[59:125].with_row_index("idx", offset=1)
mombasa = df[127:193].with_row_index("idx", offset=1)
nakuru = df[195:261].with_row_index("idx", offset=1)
kisumu = df[263:329].with_row_index("idx", offset=1)
eldoret = df[331:397].with_row_index("idx", offset=1)
kenya

In [ ]:
# df.write_excel(r"C:\Users\23225959\OneDrive - Airtel Africa\Attachments\KE_QoS_Report_20260211.xlsx", worksheet="Charting")

In [ ]:
kpi_names = kenya[kenya.columns[2]].to_list()

overrides = {
    '2G RNA': (99.93, '>'),
    '3G RNA': (99.93, '>'),
    '4G RNA': (99.93, '>'),
    '5G RNA': (99.93, '>'),

    '2G DCR': (0.3, '<='),
    '3G CS DROP': (0.15, '<='),     
    '3G PS/HS DROP': (0.3, '<='),       
    'Service Drop Rate (All) (4G)': (0.10, '<='),
    'VoLTE Drop Rate (QC1)': (0.13, '<='),
    '5G Session Drop Rate': (0.15, '<='),

    '2G Utilization ': (80.0, '<='),  
    '3G Utilization ': (80.0, '<='),
    '4G Utilization': (80.0, '<='),
    '5G Utilization': (80.0, '<='),

    '2G Voice Traffic (Total traffic/daily)': (None, None),
    '3G Voice Traffic (Total traffic/daily)': (None, None),
    'VoLTE Voice Traffic (Total traffic/daily)': (None, None),

    '2G Data Volume (GB) (Total Volume/daily)': (None, None),
    '3G Data Volume (GB) (Total Volume/daily)': (None, None),
    '4G Data Volume (GB) (Total Volume/daily)': (None, None),
    'VoLTE Data Volume (GB) (Total Volume/daily)': (None, None),
    '5G Data Volume (GB) (Total Volume/daily)': (None, None),

    '2G CSSR': (99.5, '>='),
    '3G CSSR': (99.5, '>='),
    '3G HS/PS CSSR': (99.0, '>='),
    'VoLTE CSSR': (99.5, '>='),
    '2G PSR': (90.0, '>='),
    '3G PSR': (90.0, '>='),
    '4G PSR': (90.0, '>='),
    'CSFB success rate (4G)': (99.90, '>='),
    'Session Success Rate (4G)': (99.5, '>='),
    'ERAB Setup Success Rate (QCI 1) -VoLTE': (99.5, '>='),
    'SRVCC Success Rate (3G & 2G)': (97.5, '>='),
    '5G Session Setup Success Rate': (None, None),

    'SgNb Addition Success Rate': (99.5, '>='),
    '3G Throughput': (2500, '>='), 
    '4G Throughput(DL_User_throughput) (Mbps)': (10.0, '>='), 
    '5G DL Throughput (Mbps)': (20.0, '>='), 
    '5G UL Throughput (Mbps)': (5.0, '>='),  
    '4G CQI': (8, '>='),
    '5G CQI': (8, '>='),

    '2G_TCH_Availability': (99.5, '>='),
    '2G_TCH_Blocking': (0.15, '<='),
    '2G voice Data integrity(%)': (99.90, '>='),
    '3G CS Data integrity(%)': (99.90, '>='),
    '3G PS Data integrity(%)': (99.90, '>='),
    '4G Data integrity(%)': (99.90, '>='),
    '5G Data integrity(%)': (99.90, '>='),

    '% of cells having 3G user throughput greater than 1000kbps': (85.00, '>='),
    '% of cells having 4G user throughput greater than 5Mbps ': (90.00, '>='),
    '% of cells having 5G user throughput (>=20Mbps)': (70.00, '>='),
    '% of cells with 2G RX Qual Samples 0-5(>=90%)': (90.00, '>='),

    '5G Latency': (30.0, '<'),              
    '5G Packet Loss': (0.01, '<'),           
    'Percentage_Cells_with_4G_User_Throughput_less_than_3Mbps': (5.0, '<'),
    'SD Blocking ': (0.30, '<'),           
    'No. of cells having 4G user throughput <3Mbps ': (None, None), 
}

THRESHOLDS = [(k, *(overrides.get(k, (None, None)))) for k in kpi_names]

## Charting Kenya

In [ ]:
kenya_data = kenya[1:, -30:]
kenya_header = kenya_data.columns

In [ ]:
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline
import numpy as np

# Set seaborn style
sns.set_theme(style="whitegrid", palette="husl")
sns.set_context("notebook", font_scale=1.1)

# Extract data
row_index = 0
row = kenya_data[row_index].to_dict()
threshold_dict = {kpi: (threshold, sign) for kpi, threshold, sign in THRESHOLDS}
kpis = list(row.keys())
values = [float(val.item() if hasattr(val, 'item') else val[0]) for val in row.values()]

# Smooth interpolation
x = np.array(range(len(kpis)))
y = np.array(values)
x_smooth = np.linspace(x.min(), x.max(), 300)
y_smooth = make_interp_spline(x, y, k=3)(x_smooth)

# Plot with seaborn styling
fig, ax = plt.subplots(figsize=(16, 7))
ax.plot(x_smooth, y_smooth, linewidth=3, color='#2E86AB', alpha=0.8)

# Add threshold line
first_kpi = kpi_names[row_index]
threshold, sign = threshold_dict.get(first_kpi, (None, None))
if threshold is not None:
    ax.axhline(y=threshold, color='blue', linestyle=':', linewidth=2, alpha=0.7, label=f'Threshold: {threshold}')

# Styling
ax.set_xticks(x)
ax.set_xticklabels(kpis, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Value', fontsize=12, fontweight='bold')

# Set y-axis limits to show variation
y_min, y_max = min(values), max(values)
y_range = y_max - y_min
padding = y_range * 0.2 if y_range > 0 else 0.1
ax.set_ylim(y_min - padding, y_max + padding)

ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Title with threshold info
title = f"{first_kpi} {sign} {threshold}" if threshold and sign else first_kpi
ax.set_title(title, fontsize=14, fontweight='bold', pad=20)

if threshold is not None:
    ax.legend(loc='best', frameon=True, fancybox=True, shadow=True)

plt.tight_layout()
plt.show()

In [ ]:
import os
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as ExcelImage

# Create directory structure
os.makedirs('kenya/2026-02-11', exist_ok=True)

# Prepare Nairobi data
nairobi_data = nairobi[1:, -30:]
threshold_dict = {kpi: (threshold, sign) for kpi, threshold, sign in THRESHOLDS}

# Set seaborn style
sns.set_theme(style="whitegrid", palette="husl")
sns.set_context("notebook", font_scale=1.1)

# Track which KPIs don't meet threshold and their image paths
failed_kpis = []
image_paths = []

# Loop through each KPI
for row_index in range(len(nairobi_data)):
    kpi_name = kpi_names[row_index]
    threshold, sign = threshold_dict.get(kpi_name, (None, None))
    
    # Skip if no threshold defined
    if threshold is None or sign is None:
        continue
    
    # Get data for this KPI
    row = nairobi_data[row_index].to_dict()
    kpis = list(row.keys())
    values = [float(val.item() if hasattr(val, 'item') else val[0]) for val in row.values()]
    
    # Check if any value doesn't meet threshold
    fails_threshold = False
    if sign == '>':
        fails_threshold = any(v <= threshold for v in values)
    elif sign == '>=':
        fails_threshold = any(v < threshold for v in values)
    elif sign == '<':
        fails_threshold = any(v >= threshold for v in values)
    elif sign == '<=':
        fails_threshold = any(v > threshold for v in values)
    
    if not fails_threshold:
        continue
    
    # Create plot for this KPI
    x = np.array(range(len(kpis)))
    y = np.array(values)
    x_smooth = np.linspace(x.min(), x.max(), 300)
    y_smooth = make_interp_spline(x, y, k=3)(x_smooth)
    
    fig, ax = plt.subplots(figsize=(16, 7))
    ax.plot(x_smooth, y_smooth, linewidth=3, color='#2E86AB', alpha=0.8)
    
    # Add threshold line
    ax.axhline(y=threshold, color='blue', linestyle=':', linewidth=2, alpha=0.7, label=f'Threshold: {threshold}')
    
    # Styling
    ax.set_xticks(x)
    ax.set_xticklabels(kpis, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Value', fontsize=12, fontweight='bold')
    
    # Set y-axis limits to include both data AND threshold
    y_min_data, y_max_data = min(values), max(values)
    y_min = min(y_min_data, threshold)
    y_max = max(y_max_data, threshold)
    y_range = y_max - y_min
    padding = y_range * 0.15 if y_range > 0 else 0.1
    ax.set_ylim(y_min - padding, y_max + padding)
    
    ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    title = f"Nairobi: {kpi_name} {sign} {threshold}"
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc='best', frameon=True, fancybox=True, shadow=True)
    
    plt.tight_layout()
    
    # Save plot
    safe_kpi_name = kpi_name.replace('/', '-').replace('\\', '-').replace(':', '-')
    image_path = f'kenya/2026-02-11/{safe_kpi_name}.png'
    plt.savefig(image_path, bbox_inches='tight', dpi=150)
    plt.close()
    
    failed_kpis.append(kpi_name)
    image_paths.append(image_path)
    
print(f"Created {len(failed_kpis)} plots for KPIs not meeting thresholds")

# Load Excel file and add images to Charting sheet
excel_path = "KE_QoS_Report_20260211.xlsx"
wb = load_workbook(excel_path)

# Create or get Charting sheet
if 'Charting' in wb.sheetnames:
    ws = wb['Charting']
    ws.delete_rows(1, ws.max_row)  # Clear existing content
else:
    ws = wb.create_sheet('Charting')

# Add images to sheet
current_row = 1
for i, (kpi, img_path) in enumerate(zip(failed_kpis, image_paths)):
    # Add KPI name
    ws.cell(row=current_row, column=1, value=kpi)
    current_row += 1
    
    # Add image
    img = ExcelImage(img_path)
    img.width = 960  # Adjust size
    img.height = 420
    ws.add_image(img, f'A{current_row}')
    
    # Move to next position (skip rows for image height)
    current_row += 25

# Save workbook
wb.save(excel_path)
print(f"Saved {len(failed_kpis)} plots to '{excel_path}' in 'Charting' sheet")

In [ ]:
# Analyze each plotted KPI to verify threshold logic
analysis_results = []

# Prepare Nairobi data
nairobi_data = nairobi[1:, -30:]
threshold_dict = {kpi: (threshold, sign) for kpi, threshold, sign in THRESHOLDS}

# Loop through each KPI that was plotted
for i, (kpi_name, img_path) in enumerate(zip(failed_kpis, image_paths)):
    # Find row index for this KPI
    row_index = kpi_names.index(kpi_name)
    threshold, sign = threshold_dict.get(kpi_name, (None, None))
    
    # Get data for this KPI
    row = nairobi_data[row_index].to_dict()
    values = [float(val.item() if hasattr(val, 'item') else val[0]) for val in row.values()]
    
    # Check threshold violations
    min_val = min(values)
    max_val = max(values)
    
    violations = []
    if sign == '>':
        violations = [v for v in values if v <= threshold]
    elif sign == '>=':
        violations = [v for v in values if v < threshold]
    elif sign == '<':
        violations = [v for v in values if v >= threshold]
    elif sign == '<=':
        violations = [v for v in values if v > threshold]
    
    result = {
        'kpi': kpi_name,
        'threshold': threshold,
        'sign': sign,
        'min_value': min_val,
        'max_value': max_val,
        'num_violations': len(violations),
        'violations': violations[:5] if len(violations) <= 5 else violations[:5],  # Show first 5
        'image_path': img_path
    }
    
    analysis_results.append(result)

# Print analysis
for result in analysis_results:
    print(f"\n{'='*80}")
    print(f"KPI: {result['kpi']}")
    print(f"Threshold: {result['sign']} {result['threshold']}")
    print(f"Data range: [{result['min_value']:.4f}, {result['max_value']:.4f}]")
    print(f"Number of violations: {result['num_violations']}")
    if result['num_violations'] > 0:
        print(f"Sample violations: {[f'{v:.4f}' for v in result['violations']]}")
        
        # Check if threshold is within data range (for visible threshold line)
        if result['min_value'] <= result['threshold'] <= result['max_value']:
            print("✓ Threshold line should be visible in plot")
        else:
            print("⚠ WARNING: Threshold line may not be visible (outside data range)")
    else:
        print("⚠ WARNING: No violations found - this should not be plotted!")
    print(f"Image: {result['image_path']}")
